# Phi-4 Mini ONNX មគ្គុទេសក៍​សម្រាប់​ការ​ហៅ​មុខងារ​ពហុ

This notebook demonstrates how to use Phi-4 Mini with ONNX Runtime GenAI for parallel function calling. Function calling allows the model to intelligently invoke external tools and APIs based on user requests.

## ទិដ្ឋភាពទូទៅ

In this tutorial, you'll learn how to:
- រៀបចំ Phi-4 Mini ជាមួយ ONNX Runtime GenAI
- កំណត់ស្កីម៉ាមុខងារ​សម្រាប់ការកក់យន្តហោះ និងសណ្ឋាគារ
- ប្រើការបង្កើតដោយមានមគ្គុទេសក៍ជាមួយវាក្យសម្ព័ន្ធ Lark សម្រាប់លទ្ធផលបែបរចនាសម្ព័ន្ធ
- អនុវត្តការហៅមុខងារផ្សេងៗគ្នាជាមួយគ្នាទៅក្នុងពេលតែមួយសម្រាប់ករណីកក់ទេសចរណ៍ស្មុគស្មាញ

## តម្រូវការមុនការ

Before running this notebook, ensure you have:
- បានទាញយកម៉ូដែល Phi-4 Mini ONNX
- បានដំឡើងកញ្ចប់ `onnxruntime-genai`
- ចំណេះដឹងមូលដ្ឋានអំពីគំនិតនៃការហៅមុខងារ


## ជំហាន 1: នាំចូលបណ្ណាល័យដែលចាំបាច់

ជាដំបូង យើងនឹងនាំចូលបណ្ណាល័យដែលចាំបាច់សម្រាប់ការអនុវត្តន៍ការហៅមុខងាររបស់យើង។


In [1]:
import json

In [2]:
import onnxruntime_genai as og

## ជំហានទី 2: ការរៀបចំនិងកំណត់រចនាសម្ព័ន្ធម៉ូឌែល

ឥឡូវនេះយើងនឹងកំណត់រចនាសម្ព័ន្ធសម្រាប់ម៉ូឌែល Phi-4 Mini ONNX។ សូមប្រាកដប្តូរផ្លូវទៅកាន់ថតដែលមានម៉ូឌែលពិតរបស់អ្នក។


In [ ]:
# TODO: Replace with your actual Phi-4 Mini ONNX model path
# Download from: https://huggingface.co/microsoft/Phi-4-mini-onnx
path = 'Your phi-4-mini-onnx path'  # Update this path!

In [4]:
config = og.Config(path)

In [5]:
model = og.Model(config)

In [6]:
tokenizer = og.Tokenizer(model)
tokenizer_stream = tokenizer.create_stream()

## ជំហានទី 3: កំណត់ប៉ារ៉ាម៉ែត្រការបង្កើត

កំណត់ប៉ារ៉ាម៉ែត្រក្នុងការបង្កើត ដើម្បីគ្រប់គ្រងអាកប្បកិរិយានៃលទ្ធផលដែលម៉ូដែលបញ្ចេញ។ ការកំណត់ទាំងនេះធានាថាការឆ្លើយតបមានលទ្ធផលថេរ និងផ្ដោតសម្រាប់ការហៅមុខងារ។


In [7]:
# Configure generation parameters for deterministic function calling
search_options = {}
search_options['max_length'] = 4096      # Maximum tokens to generate
search_options['temperature'] = 0.00001  # Very low temperature for deterministic output
search_options['top_p'] = 1.0            # Nucleus sampling parameter
search_options['do_sample'] = False       # Disable sampling for consistent results

## ជំហាន 4: កំណត់អនុគមន៍ដែលអាចប្រើបាន

នៅទីនេះ យើងកំណត់អនុគមន៍ដែលជំនួយការ AI របស់យើងអាចហៅបាន។ នៅក្នុងឧទាហរណ៍នេះ យើងមានអនុគមន៍ទាក់ទងនឹងការធ្វើដំណើរពីរ៖
1. **booking_flight_tickets**: សម្រាប់កក់សំបុត្រយន្តហោះរវាងព្រលានយន្តហោះ
2. **booking_hotels**: សម្រាប់កក់បន្ទប់សណ្ឋាគារ

ការកំណត់អនុគមន៍ទាំងនេះអនុវត្តតាមទ្រង់ទ្រាយ function calling schema របស់ OpenAI.


In [8]:
tool_list = '[{"name": "booking_flight_tickets", "description": "booking flights", "parameters": {"origin_airport_code": {"description": "The name of Departure airport code", "type": "string"}, "destination_airport_code": {"description": "The name of Destination airport code", "type": "string"}, "departure_date": {"description": "The date of outbound flight", "type": "string"}, "return_date": {"description": "The date of return flight", "type": "string"}}}, {"name": "booking_hotels", "description": "booking hotel", "parameters": {"destination": {"description": "The name of the city", "type": "string"}, "check_in_date": {"description": "The date of check in", "type": "string"}, "checkout_date": {"description": "The date of check out", "type": "string"}}}]'

## ជំហានទី 5: អនុគមន៍ជំនួយសម្រាប់បង្កើតវេយ្យាករណ៍

អនុគមន៍ជំនួយទាំងនេះបម្លែងការកំណត់អនុគមន៍របស់យើងទៅជាទ្រង់ទ្រាយវេយ្យាករណ៍ Lark ដែលត្រូវបានប្រើដោយ ONNX Runtime GenAI សម្រាប់ការបង្កើតដោយមានការណែនាំ។ នេះធានាថាម៉ូដែលនឹងបញ្ចេញការហៅអនុគមន៍ដែលមានសុពលភាពក្នុងទ្រង់ទ្រាយ JSON ដែលត្រឹមត្រូវ។


In [9]:
def get_lark_grammar(input_tools):
    tools_list = get_tools_list(input_tools)
    prompt_tool_input = create_prompt_tool_input(tools_list)
    if len(tools_list) == 1:
        # output = ("start: TEXT | fun_call\n" "TEXT: /[^{](.|\\n)*/\n" " fun_call: <|tool_call|> %json " + json.dumps(tools_list[0]))
        output = ("start: TEXT | fun_call\n" "TEXT: /[^{](.|\\n)*/\n" " fun_call: <|tool_call|> %json " + json.dumps(convert_tool_to_grammar_input(tools_list[0])))
        return prompt_tool_input, output
    else:
        return prompt_tool_input, "start: TEXT | fun_call \n TEXT: /[^{](.|\n)*/ \n fun_call: <|tool_call|> %json {\"anyOf\": [" + ','.join([json.dumps(tool) for tool in tools_list]) + "]}"


In [10]:
def get_tools_list(input_tools):
    # input_tools format: '[{"name": "fn1", "description": "fn details", "parameters": {"p1": {"description": "details", "type": "string"}}},
    # {"fn2": 2},{"fn3": 3}]'
    tools_list = []
    try:
        tools_list = json.loads(input_tools)
    except json.JSONDecodeError:
        raise ValueError("Invalid JSON format for tools list, expected format: '[{\"name\": \"fn1\"},{\"name\": \"fn2\"}]'")
    if len(tools_list) == 0:
        raise ValueError("Tools list cannot be empty")
    return tools_list

In [11]:
def create_prompt_tool_input(tools_list):
    tool_input = str(tools_list[0])
    for tool in tools_list[1:]:
        tool_input += ',' + str(tool)
    return tool_input

In [12]:
def convert_tool_to_grammar_input(tool):
    param_props = {}
    required_params = []
    for param_name, param_info in tool.get("parameters", {}).items():
        param_props[param_name] = {
            "type": param_info.get("type", "string"),
            "description": param_info.get("description", "")
        }
        required_params.append(param_name)
    output_schema = {
        "description": tool.get('description', ''),
        "type": "object",
        "required": ["name", "parameters"],
        "additionalProperties": False,
        "properties": {
            "name": { "const": tool["name"] },
            "parameters": {
                "type": "object",
                "properties": param_props,
                "required": required_params,
                "additionalProperties": False
            }
        }
    }
    if len(param_props) == 0:
        output_schema["required"] = ["name"]
    return output_schema

In [13]:
get_lark_grammar(tool_list)

("{'name': 'booking_flight_tickets', 'description': 'booking flights', 'parameters': {'origin_airport_code': {'description': 'The name of Departure airport code', 'type': 'string'}, 'destination_airport_code': {'description': 'The name of Destination airport code', 'type': 'string'}, 'departure_date': {'description': 'The date of outbound flight', 'type': 'string'}, 'return_date': {'description': 'The date of return flight', 'type': 'string'}}},{'name': 'booking_hotels', 'description': 'booking hotel', 'parameters': {'destination': {'description': 'The name of the city', 'type': 'string'}, 'check_in_date': {'description': 'The date of check in', 'type': 'string'}, 'checkout_date': {'description': 'The date of check out', 'type': 'string'}}}",
 'start: TEXT | fun_call \n TEXT: /[^{](.|\n)*/ \n fun_call: <|tool_call|> %json {"anyOf": [{"name": "booking_flight_tickets", "description": "booking flights", "parameters": {"origin_airport_code": {"description": "The name of Departure airport c

## ជំហានទី 6: សាកល្បងការបង្កើតវាក្យសម្ព័ន្ធ

យើង​សាកល្បង​អនុគមន៍​បង្កើត​វាក្យសម្ព័ន្ធ​របស់​យើង​ដើម្បី​មើល​ថា​វា​បម្លែង​ការកំណត់ឧបករណ៍​របស់​យើង​ទៅជា​ទ្រង់ទ្រាយ​ដែល​ត្រឹមត្រូវ​យ៉ាងដូចម្តេច។


In [14]:
prompt_tool_input, guidance_input = get_lark_grammar(tool_list)

## ជំហានទី 7: រៀបចំ System Prompt និង Generator

ឥឡូវនេះយើងនឹងបង្កើត System Prompt ដែលប្រាប់ម៉ូឌែលអំពីឧបករណ៍ដែលអាចប្រើបាន និងរៀបចំ Generator ជាមួយប៉ារ៉ាម៉ែត្រណែនាំសម្រាប់ការបង្កើត។


In [15]:
# Define the system prompt that introduces the assistant and its capabilities
system_prompt = "You are a helpful assistant with these tools."

In [16]:
# Format the system message with tools information
messages = f"""[{{"role": "system", "content": "{system_prompt}", "tools": "{prompt_tool_input}"}}]"""

In [17]:
# Apply chat template to format the system prompt properly
tokenizer_input_system_prompt = tokenizer.apply_chat_template(messages=messages, add_generation_prompt=False)

In [18]:
tokenizer_input_system_prompt

"<|system|>You are a helpful assistant with these tools.<|tool|>{'name': 'booking_flight_tickets', 'description': 'booking flights', 'parameters': {'origin_airport_code': {'description': 'The name of Departure airport code', 'type': 'string'}, 'destination_airport_code': {'description': 'The name of Destination airport code', 'type': 'string'}, 'departure_date': {'description': 'The date of outbound flight', 'type': 'string'}, 'return_date': {'description': 'The date of return flight', 'type': 'string'}}},{'name': 'booking_hotels', 'description': 'booking hotel', 'parameters': {'destination': {'description': 'The name of the city', 'type': 'string'}, 'check_in_date': {'description': 'The date of check in', 'type': 'string'}, 'checkout_date': {'description': 'The date of check out', 'type': 'string'}}}<|/tool|><|end|><|endoftext|>"

In [19]:
input_tokens = tokenizer.encode(tokenizer_input_system_prompt)

In [20]:
input_tokens = input_tokens[:-1]

In [21]:
system_prompt_length = len(input_tokens)

## ជំហានទី 8: ចាប់ផ្តើម Generator ជាមួយការបង្កើតដោយមានការណែនាំ

ឥឡូវនេះ យើងនឹងបង្កើត Generator ជាមួយប៉ារ៉ាម៉ែត្រដែលបានកំណត់ ហើយអនុវត្តវេយ្យាករណ៍ Lark សម្រាប់ការបង្កើតដែលមានការណែនាំ។


In [22]:
# Create generator parameters and apply search options
params = og.GeneratorParams(model)
params.set_search_options(**search_options)

In [23]:
# Apply Lark grammar for guided generation to ensure valid function call format
params.set_guidance("lark_grammar", guidance_input)

In [24]:
generator = og.Generator(model, params)

In [25]:
generator.append_tokens(input_tokens)

## ជំហាន 9: សាកល្បងការហៅមុខងារដោយស្របពេល

ឥឡូវនេះ យើងនឹងសាកល្បងការរៀបចំបញ្ចូលរបស់យើងជាមួយស្ថានការណ៍​កក់​ទេសចរណ៍​ស្មុគស្មាញ ដែលតម្រូវឱ្យហៅមុខងារច្រើន។


In [26]:
# Complex travel booking request that requires both flight and hotel booking
text = "book flight ticket from Beijing to Paris(using airport code) in 2025-12-04 to 2025-12-10 , then book hotel from 2025-12-04 to 2025-12-10 in Paris"

In [27]:
# Format user message and apply chat template
messages = f"""[{{"role": "user", "content": "{text}"}}]"""

# Apply Chat Template for user input
user_prompt = tokenizer.apply_chat_template(messages=messages, add_generation_prompt=True)
input_tokens = tokenizer.encode(user_prompt)
generator.append_tokens(input_tokens)

In [28]:
user_prompt

'<|user|>book flight ticket from Beijing to Paris(using airport code) in 2025-12-04 to 2025-12-10 , then book hotel from 2025-12-04 to 2025-12-10 in Paris<|end|><|assistant|>'

### មើលសំណើអ្នកប្រើដែលបានទ្រង់ទ្រាយ


### បង្កើតការហៅមុខងារ

ម៉ូឌែល​នឹងឥឡូវនេះបង្កើតការហៅមុខងារដែលមានរចនាសម្ព័ន្ធ​តាមសំណើររបស់អ្នកប្រើ។ អរគុណដល់ការបង្កើតដែលមានការណែនាំ លទ្ធផលនឹងមានទ្រង់ទ្រាយ JSON ត្រឹមត្រូវ ដែលអាចអនុវត្តបានដោយផ្ទាល់។

**លទ្ធផលដែលគេរំពឹងទុក**: ម៉ូឌែលគួរតែបង្កើតការហៅមុខងារសម្រាប់:
1. `booking_flight_tickets` ជាមួយព័ត៌មានលម្អិតពី Beijing (PEK) ទៅ Paris (CDG)
2. `booking_hotels` ជាមួយព័ត៌មានលម្អិតអំពីការស្នាក់នៅនៅ Paris

រត់កោសិកាខាងក្រោមដើម្បីមើលការបង្កើតបន្តផ្ទាល់:


In [29]:
# Generate tokens one by one and stream the output
while not generator.is_done():
    generator.generate_next_token()
    new_token = generator.get_next_tokens()[0]
    print(tokenizer_stream.decode(new_token), end='', flush=True)

[{"name": "booking_flight_tickets", "arguments": {"origin_airport_code": "PEKK", "destination_airport_code": "CDG", "departure_date": "2025-12-04", "return_date": "2025-12-10"}}, {"name": "booking_hotels", "arguments": {"destination": "Paris", "check_in_date": "2025-12-04", "checkout_date": "2025-12-10"}}]

## សេចក្តីសន្និដ្ឋាន

🎉 **អបអរសាទរ!** អ្នកបានអនុវត្តការហៅមុខងារជាស្របពេល (parallel function calling) ជាមួយ Phi-4 Mini ដោយប្រើ ONNX Runtime GenAI បានដោយជោគជ័យ។

### អ្វីដែលអ្នកបានរៀន៖

1. **ការកំណត់ម៉ូដែល**: របៀបកំណត់ Phi-4 Mini ជាមួយ ONNX Runtime GenAI
2. **ការបញ្ជាក់មុខងារ**: របៀបកំណត់ស្កីម៉ាសម្រាប់ការហៅមុខងារ AI
3. **ការបង្កើតដោយណែនាំ**: របៀបប្រើវេយ្យាករណ៍ Lark សម្រាប់បង្កើតលទ្ធផលដែលមានរចនាសម្ព័ន្ធ
4. **ការហៅមុខងារជាស្របពេល**: របៀបដោះស្រាយសំណើស្មុគស្មាញដែលទាមទារការហៅមុខងារជាច្រើន

### អត្ថប្រយោជន៍សំខាន់ៗ:

- ✅ **លទ្ធផលដែលមានរចនាសម្ព័ន្ធ**: ការបង្កើតដោយណែនាំធានាថាការហៅមុខងារប្រភេទ JSON មានសុពលភាព
- ✅ **ដំណើរការជាស្របពេល**: ដោះស្រាយការហៅមុខងារជាច្រើនក្នុងសំណើតែមួយ
- ✅ **ប្រសិទ្ធភាពខ្ពស់**: ONNX Runtime ផ្តល់នូវការសន្និដ្ឋានដែលបានបង្កើតអោយប្រសើរ
- ✅ **ស្កីម៉ាដែលបត់បែនបាន**: ងាយស្រួលក្នុងការបន្ថែម ឬ កែប្រែការបញ្ជាក់មុខងារ

### ធនធាន:

- [ឯកសារណែនាំ Phi-4 Mini](https://huggingface.co/microsoft/Phi-4-mini-onnx)
- [ឯកសារ ONNX Runtime GenAI](https://onnxruntime.ai/docs/genai/)
- [ការអនុវត្តល្អបំផុតសម្រាប់ការហៅមុខងារ](https://platform.openai.com/docs/guides/function-calling)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែដោយ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខិតខំស្វែងរកភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាការបកប្រែស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាដើមគួរត្រូវបានទទួលស្គាល់ថាជាប្រភពដែលគួរឱ្យទុកចិត្ត។ សម្រាប់ព័ត៌មានសំខាន់ៗ គួរតែពិចារណាការបកប្រែដោយអ្នកមានវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសណាមួយដែលកើតមានពីការប្រើប្រាស់ការបកប្រែនេះ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
